# Bước 6 – Suy Luận & Cập Nhật (Inference)

## Mục tiêu
Xây dựng **pipeline suy luận tự động** có khả năng:
1. Tự động tải dữ liệu mới nhất sau khi kết thúc phiên giao dịch
2. Tính toán lại tất cả features (technical indicators, lag features, volatility...)
3. Chạy mô hình để **dự báo xu hướng phiên giao dịch tiếp theo (t+1)**
4. Xuất kết quả ra file `.csv` và hiển thị trên màn hình

## Pipeline Inference

```
Dữ liệu mới (vnstock/yfinance)
        ↓
Tính Technical Indicators (SMA, EMA, RSI, MACD, BB)
        ↓
Tính Lag Features + Volatility
        ↓
Tích hợp VN-Index & USD/VND returns
        ↓
Scale features (cho Logistic Regression)
        ↓
Chạy model → Dự báo t+1
        ↓
Xuất kết quả (CSV + hiển thị)
```

## 6.1. Import thư viện và cấu hình

In [ ]:
import os
import pickle
import warnings
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thư viện tải dữ liệu chứng khoán
from vnstock3 import Vnstock
import yfinance as yf

# Cấu hình
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

# Đường dẫn
MODELS_DIR = os.path.join('..', 'models')
OUTPUTS_DIR = os.path.join('..', 'outputs')
DATA_DIR = os.path.join('..', 'data')
os.makedirs(OUTPUTS_DIR, exist_ok=True)

STOCK_SYMBOLS = ['VCB', 'FPT', 'HPG', 'VIC', 'POW']
TODAY = datetime.now().strftime('%Y-%m-%d')

print(f"✓ Import thành công")
print(f"Ngày chạy inference: {TODAY}")
print(f"Mã cổ phiếu: {STOCK_SYMBOLS}")

In [ ]:
# === TẢI MODELS, SCALERS, METADATA ĐÃ HUẤN LUYỆN ===
models = {}
scalers = {}

CLF_MODELS = ['LogisticRegression', 'RandomForest_Cls', 'XGBoost_Cls']
REG_MODELS = ['RandomForest_Reg', 'XGBoost_Reg']

for symbol in STOCK_SYMBOLS:
    models[symbol] = {}
    stock_dir = os.path.join(MODELS_DIR, symbol)
    
    # Tải scaler
    with open(os.path.join(stock_dir, f'{symbol}_feature_scaler.pkl'), 'rb') as f:
        scalers[symbol] = pickle.load(f)
    
    # Tải tất cả models
    for model_name in CLF_MODELS + REG_MODELS:
        with open(os.path.join(stock_dir, f'{symbol}_{model_name}.pkl'), 'rb') as f:
            models[symbol][model_name] = pickle.load(f)
    
    print(f"✓ {symbol}: scaler + {len(CLF_MODELS) + len(REG_MODELS)} models")

# Tải metadata
with open(os.path.join(MODELS_DIR, 'training_metadata.pkl'), 'rb') as f:
    training_meta = pickle.load(f)

feature_cols = training_meta['feature_cols']

# Tải evaluation results (để biết model tốt nhất)
eval_path = os.path.join(MODELS_DIR, 'evaluation_results.pkl')
if os.path.exists(eval_path):
    with open(eval_path, 'rb') as f:
        eval_results = pickle.load(f)
    best_cls_models = eval_results.get('best_cls_models', {})
    best_reg_models = eval_results.get('best_reg_models', {})
    print(f"\n✓ Best models loaded from evaluation:")
    for s in STOCK_SYMBOLS:
        print(f"  {s}: Cls={best_cls_models.get(s, 'XGBoost_Cls')}, Reg={best_reg_models.get(s, 'XGBoost_Reg')}")
else:
    # Mặc định dùng XGBoost nếu chưa có evaluation
    best_cls_models = {s: 'XGBoost_Cls' for s in STOCK_SYMBOLS}
    best_reg_models = {s: 'XGBoost_Reg' for s in STOCK_SYMBOLS}
    print("⚠ Chưa có evaluation results, dùng XGBoost làm mặc định")

print(f"\nFeature columns ({len(feature_cols)}): {feature_cols}")

## 6.2. Hàm tải dữ liệu mới nhất

Tự động tải dữ liệu OHLCV mới nhất từ `vnstock` (fallback `yfinance`) cho các mã cổ phiếu, VN-Index và USD/VND. Cần ít nhất 60 phiên gần nhất để tính đủ các technical indicators (SMA_50 cần 50 giá trị).

In [ ]:
def fetch_latest_data(symbol: str, lookback_days: int = 120) -> pd.DataFrame:
    """
    Tải dữ liệu OHLCV mới nhất cho 1 mã cổ phiếu.
    
    Parameters:
        symbol: Mã cổ phiếu (VD: 'FPT', 'VCB', 'VNINDEX')
        lookback_days: Số ngày lịch sử cần tải (mặc định 120 ngày ~ 80 phiên giao dịch)
    
    Returns:
        DataFrame với cột: time, open, high, low, close, volume
    """
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=lookback_days)).strftime('%Y-%m-%d')
    
    try:
        # Nguồn chính: vnstock
        stock = Vnstock().stock(symbol=symbol, source='VCI')
        df = stock.quote.history(start=start_date, end=end_date, interval='1D')
        df.columns = df.columns.str.lower()
        
        if 'time' in df.columns:
            df['time'] = pd.to_datetime(df['time'])
        elif 'date' in df.columns:
            df.rename(columns={'date': 'time'}, inplace=True)
            df['time'] = pd.to_datetime(df['time'])
        
        required_cols = ['time', 'open', 'high', 'low', 'close', 'volume']
        df = df[[c for c in required_cols if c in df.columns]]
        df = df.sort_values('time').drop_duplicates(subset='time').reset_index(drop=True)
        return df
        
    except Exception as e:
        print(f"  ⚠ vnstock failed for {symbol}: {e}")
        print(f"  → Trying yfinance fallback...")
        
        try:
            ticker_symbol = f"{symbol}.VN" if symbol not in ['VNINDEX'] else '^VNINDEX'
            ticker = yf.Ticker(ticker_symbol)
            df = ticker.history(start=start_date, end=end_date)
            df = df.reset_index()
            df.columns = df.columns.str.lower()
            df.rename(columns={'date': 'time'}, inplace=True)
            df = df[['time', 'open', 'high', 'low', 'close', 'volume']]
            df['time'] = pd.to_datetime(df['time']).dt.tz_localize(None)
            df = df.sort_values('time').drop_duplicates(subset='time').reset_index(drop=True)
            return df
        except Exception as e2:
            print(f"  ✗ Both sources failed for {symbol}: {e2}")
            return None


def fetch_usdvnd(lookback_days: int = 120) -> pd.DataFrame:
    """Tải tỷ giá USD/VND mới nhất từ yfinance."""
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=lookback_days)).strftime('%Y-%m-%d')
    
    try:
        usdvnd = yf.Ticker("VND=X")
        df = usdvnd.history(start=start_date, end=end_date)
        df = df.reset_index()
        df.columns = df.columns.str.lower()
        df.rename(columns={'date': 'time'}, inplace=True)
        df['time'] = pd.to_datetime(df['time']).dt.tz_localize(None)
        df = df[['time', 'open', 'high', 'low', 'close', 'volume']]
        df = df.sort_values('time').drop_duplicates(subset='time').reset_index(drop=True)
        return df
    except Exception as e:
        print(f"  ✗ Failed to fetch USD/VND: {e}")
        return None

print("✓ Các hàm fetch data đã sẵn sàng")

## 6.3. Hàm tính toán Features

Tái tạo lại toàn bộ quá trình Feature Engineering (Bước 3) cho dữ liệu mới:
- **Technical Indicators**: SMA, EMA, RSI, MACD, Bollinger Bands
- **Lag Features**: close_lag_1/3/7, return_lag_1/3/7
- **Volatility**: rolling std 10 phiên
- **Volume Features**: volume_sma_10, volume_ratio
- **Market Features**: vnindex_return, usdvnd_return

In [ ]:
def compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    """Tính Relative Strength Index (RSI)."""
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = (-delta).where(delta < 0, 0.0)
    avg_gain = gain.rolling(window=period, min_periods=period).mean()
    avg_loss = loss.rolling(window=period, min_periods=period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))


def compute_features(df: pd.DataFrame, vnindex_df: pd.DataFrame = None, 
                      usdvnd_df: pd.DataFrame = None) -> pd.DataFrame:
    """
    Tính toàn bộ features cho dữ liệu OHLCV.
    Tái tạo đúng pipeline Feature Engineering (Bước 3).
    
    Parameters:
        df: DataFrame OHLCV (time, open, high, low, close, volume)
        vnindex_df: DataFrame VN-Index OHLCV (optional)
        usdvnd_df: DataFrame USD/VND OHLCV (optional)
    
    Returns:
        DataFrame với đầy đủ feature columns (dòng cuối = phiên gần nhất)
    """
    df = df.copy().sort_values('time').reset_index(drop=True)
    
    # === 1. TECHNICAL INDICATORS ===
    # SMA (Simple Moving Average)
    df['SMA_10'] = df['close'].rolling(window=10).mean()
    df['SMA_20'] = df['close'].rolling(window=20).mean()
    df['SMA_50'] = df['close'].rolling(window=50).mean()
    
    # EMA (Exponential Moving Average)
    df['EMA_12'] = df['close'].ewm(span=12, adjust=False).mean()
    df['EMA_26'] = df['close'].ewm(span=26, adjust=False).mean()
    
    # RSI (Relative Strength Index)
    df['RSI_14'] = compute_rsi(df['close'], period=14)
    
    # MACD (Moving Average Convergence Divergence)
    df['MACD'] = df['EMA_12'] - df['EMA_26']
    df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_histogram'] = df['MACD'] - df['MACD_signal']
    
    # Bollinger Bands
    bb_sma = df['close'].rolling(window=20).mean()
    bb_std = df['close'].rolling(window=20).std()
    df['BB_upper'] = bb_sma + 2 * bb_std
    df['BB_middle'] = bb_sma
    df['BB_lower'] = bb_sma - 2 * bb_std
    
    # === 2. DAILY RETURN ===
    df['daily_return'] = df['close'].pct_change()
    
    # === 3. LAG FEATURES ===
    for lag in [1, 3, 7]:
        df[f'close_lag_{lag}'] = df['close'].shift(lag)
        df[f'return_lag_{lag}'] = df['daily_return'].shift(lag)
    
    # === 4. VOLATILITY ===
    df['volatility_10'] = df['close'].rolling(window=10).std()
    
    # === 5. VOLUME FEATURES ===
    df['volume_sma_10'] = df['volume'].rolling(window=10).mean()
    df['volume_ratio'] = df['volume'] / df['volume_sma_10']
    
    # === 6. MARKET FEATURES (VN-Index & USD/VND) ===
    if vnindex_df is not None:
        vni = vnindex_df.copy()
        vni['time'] = pd.to_datetime(vni['time'])
        vni['vnindex_return'] = vni['close'].pct_change()
        vni = vni[['time', 'vnindex_return']]
        df = df.merge(vni, on='time', how='left')
        df['vnindex_return'] = df['vnindex_return'].ffill()
    else:
        df['vnindex_return'] = 0.0
    
    if usdvnd_df is not None:
        usd = usdvnd_df.copy()
        usd['time'] = pd.to_datetime(usd['time'])
        usd['usdvnd_return'] = usd['close'].pct_change()
        usd = usd[['time', 'usdvnd_return']]
        df = df.merge(usd, on='time', how='left')
        df['usdvnd_return'] = df['usdvnd_return'].ffill()
    else:
        df['usdvnd_return'] = 0.0
    
    return df


print("✓ Hàm compute_features đã sẵn sàng")
print(f"  Features cần tính: {len(feature_cols)} features")
print(f"  {feature_cols}")

## 6.4. Hàm Inference chính — `daily_inference()`

Hàm core của pipeline: nhận mã cổ phiếu, tự động thực hiện toàn bộ quá trình từ tải dữ liệu → tính features → chạy model → trả kết quả dự báo.

```python
def daily_inference(ticker: str, model, scaler) -> dict:
    """
    Hàm chạy suy luận hàng ngày
    Input : mã cổ phiếu, mô hình đã huấn luyện, scaler
    Output: dict chứa dự báo xu hướng và xác suất
    """
```

In [ ]:
def daily_inference(ticker: str, model, scaler) -> dict:
    """
    Hàm chạy suy luận hàng ngày.
    
    Input:
        ticker  : mã cổ phiếu (VD: 'FPT', 'VCB', 'HPG')
        model   : dict chứa tất cả models đã train cho mã này
                  {'LogisticRegression': model_obj, 'RandomForest_Cls': ..., ...}
        scaler  : StandardScaler đã fit trên training data
    
    Output:
        dict chứa dự báo xu hướng, xác suất, giá dự báo, và metadata
    """
    result = {
        'ticker': ticker,
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'status': 'SUCCESS',
        'error': None
    }
    
    try:
        # === BƯỚC 1: Tải dữ liệu mới nhất ===
        print(f"  [1/4] Tải dữ liệu {ticker}...")
        stock_df = fetch_latest_data(ticker, lookback_days=150)
        vnindex_df = fetch_latest_data('VNINDEX', lookback_days=150)
        usdvnd_df = fetch_usdvnd(lookback_days=150)
        
        if stock_df is None or len(stock_df) < 55:
            raise ValueError(f"Không đủ dữ liệu cho {ticker} (cần >= 55 phiên, có {len(stock_df) if stock_df is not None else 0})")
        
        # Lưu thông tin phiên gần nhất
        latest = stock_df.iloc[-1]
        result['latest_date'] = str(latest['time'].date()) if hasattr(latest['time'], 'date') else str(latest['time'])[:10]
        result['latest_close'] = float(latest['close'])
        result['latest_volume'] = int(latest['volume'])
        
        # === BƯỚC 2: Tính toán Features ===
        print(f"  [2/4] Tính features...")
        df_features = compute_features(stock_df, vnindex_df, usdvnd_df)
        
        # Lấy dòng cuối cùng (phiên gần nhất) — đây là input cho model
        latest_row = df_features.iloc[-1]
        
        # Kiểm tra missing features
        missing_features = [col for col in feature_cols if col not in df_features.columns]
        if missing_features:
            raise ValueError(f"Thiếu features: {missing_features}")
        
        X_latest = df_features[feature_cols].iloc[[-1]].values  # shape (1, n_features)
        
        # Kiểm tra NaN
        if np.isnan(X_latest).any():
            nan_cols = [feature_cols[i] for i in range(len(feature_cols)) if np.isnan(X_latest[0][i])]
            print(f"  ⚠ NaN detected in features: {nan_cols}")
            # Fill NaN với 0 (fallback)
            X_latest = np.nan_to_num(X_latest, nan=0.0)
        
        # Scale cho Logistic Regression
        X_latest_scaled = scaler.transform(X_latest)
        
        # === BƯỚC 3: Chạy tất cả models ===
        print(f"  [3/4] Chạy inference...")
        
        # --- Classification: Dự báo xu hướng Tăng/Giảm ---
        clf_predictions = {}
        for model_name in CLF_MODELS:
            if model_name in model:
                X = X_latest_scaled if 'Logistic' in model_name else X_latest
                clf_model = model[model_name]
                
                pred = clf_model.predict(X)[0]
                
                # Lấy xác suất nếu model hỗ trợ
                if hasattr(clf_model, 'predict_proba'):
                    proba = clf_model.predict_proba(X)[0]
                    prob_up = float(proba[1]) if len(proba) > 1 else float(proba[0])
                    prob_down = float(proba[0]) if len(proba) > 1 else 1 - float(proba[0])
                else:
                    prob_up = float(pred)
                    prob_down = 1.0 - prob_up
                
                clf_predictions[model_name] = {
                    'prediction': int(pred),
                    'trend': 'TANG' if pred == 1 else 'GIAM',
                    'prob_up': round(prob_up, 4),
                    'prob_down': round(prob_down, 4),
                    'confidence': round(max(prob_up, prob_down), 4)
                }
        
        result['classification'] = clf_predictions
        
        # --- Regression: Dự báo giá đóng cửa ngày t+1 ---
        reg_predictions = {}
        for model_name in REG_MODELS:
            if model_name in model:
                reg_model = model[model_name]
                pred_price = float(reg_model.predict(X_latest)[0])
                
                # Tính % thay đổi so với giá hiện tại
                price_change = pred_price - result['latest_close']
                price_change_pct = (price_change / result['latest_close']) * 100
                
                reg_predictions[model_name] = {
                    'predicted_price': round(pred_price, 2),
                    'price_change': round(price_change, 2),
                    'price_change_pct': round(price_change_pct, 2)
                }
        
        result['regression'] = reg_predictions
        
        # === BƯỚC 4: Tổng hợp — Dự báo best model ===
        print(f"  [4/4] Tổng hợp kết quả...")
        
        # Best classification model
        best_cls_name = best_cls_models.get(ticker, 'XGBoost_Cls')
        if best_cls_name in clf_predictions:
            result['best_cls_model'] = best_cls_name
            result['best_trend'] = clf_predictions[best_cls_name]['trend']
            result['best_confidence'] = clf_predictions[best_cls_name]['confidence']
            result['best_prob_up'] = clf_predictions[best_cls_name]['prob_up']
        
        # Best regression model
        best_reg_name = best_reg_models.get(ticker, 'XGBoost_Reg')
        if best_reg_name in reg_predictions:
            result['best_reg_model'] = best_reg_name
            result['best_predicted_price'] = reg_predictions[best_reg_name]['predicted_price']
            result['best_price_change_pct'] = reg_predictions[best_reg_name]['price_change_pct']
        
        # Ensemble vote: đa số models dự đoán gì?
        votes = [clf_predictions[m]['prediction'] for m in clf_predictions]
        result['ensemble_trend'] = 'TANG' if sum(votes) > len(votes) / 2 else 'GIAM'
        result['ensemble_agreement'] = f"{max(sum(votes), len(votes) - sum(votes))}/{len(votes)}"
        
    except Exception as e:
        result['status'] = 'ERROR'
        result['error'] = str(e)
        print(f"  ✗ Error: {e}")
    
    return result


print("✓ Hàm daily_inference() đã sẵn sàng")

## 6.5. Chạy Inference cho tất cả 5 mã cổ phiếu

Chạy pipeline dự báo phiên **t+1** cho tất cả mã: VCB, FPT, HPG, VIC, POW.

In [ ]:
# === CHẠY INFERENCE CHO TẤT CẢ 5 MÃ ===
all_results = {}

for symbol in STOCK_SYMBOLS:
    print(f"\n{'='*70}")
    print(f"INFERENCE — {symbol}")
    print(f"{'='*70}")
    
    result = daily_inference(
        ticker=symbol,
        model=models[symbol],
        scaler=scalers[symbol]
    )
    
    all_results[symbol] = result
    
    if result['status'] == 'SUCCESS':
        print(f"\n  ✓ {symbol} | Phiên gần nhất: {result['latest_date']} | Giá: {result['latest_close']:.2f}")
        print(f"    Best Model Cls: {result.get('best_cls_model', 'N/A')} → {result.get('best_trend', 'N/A')} (Confidence: {result.get('best_confidence', 0):.1%})")
        print(f"    Best Model Reg: {result.get('best_reg_model', 'N/A')} → Giá dự báo: {result.get('best_predicted_price', 0):.2f} ({result.get('best_price_change_pct', 0):+.2f}%)")
        print(f"    Ensemble Vote:  {result.get('ensemble_trend', 'N/A')} ({result.get('ensemble_agreement', 'N/A')} models đồng ý)")
    else:
        print(f"\n  ✗ {symbol}: {result['error']}")

print(f"\n{'='*70}")
print(f"✓ Inference hoàn tất cho {len(all_results)} mã cổ phiếu")
print(f"{'='*70}")

## 6.6. Hiển thị kết quả dự báo — Bảng tổng hợp

In [ ]:
# === BẢNG TỔNG HỢP DỰ BÁO PHIÊN T+1 ===
print("=" * 110)
print(f"DỰ BÁO XU HƯỚNG GIÁ CỔ PHIẾU — PHIÊN T+1 ({datetime.now().strftime('%Y-%m-%d %H:%M')})")
print("=" * 110)

# Bảng Classification (xu hướng)
print(f"\n{'Mã CP':8s} {'Phiên cuối':12s} {'Giá hiện tại':>14s} {'Xu hướng':>10s} {'Xác suất Tăng':>14s} {'Confidence':>12s} {'Ensemble':>10s} {'Model':>20s}")
print("-" * 110)

for symbol in STOCK_SYMBOLS:
    r = all_results[symbol]
    if r['status'] == 'SUCCESS':
        print(f"{symbol:8s} {r['latest_date']:12s} {r['latest_close']:14.2f} "
              f"{r.get('best_trend', 'N/A'):>10s} {r.get('best_prob_up', 0):14.2%} "
              f"{r.get('best_confidence', 0):12.2%} {r.get('ensemble_trend', 'N/A'):>10s} "
              f"{r.get('best_cls_model', 'N/A'):>20s}")
    else:
        print(f"{symbol:8s} {'ERROR':12s} {'N/A':>14s} {'N/A':>10s} {'N/A':>14s} {'N/A':>12s} {'N/A':>10s} {'N/A':>20s}")

# Bảng Regression (giá dự báo)
print(f"\n{'Mã CP':8s} {'Giá hiện tại':>14s} {'Giá dự báo t+1':>16s} {'Thay đổi':>10s} {'%':>8s} {'Model':>20s}")
print("-" * 80)

for symbol in STOCK_SYMBOLS:
    r = all_results[symbol]
    if r['status'] == 'SUCCESS':
        pred_price = r.get('best_predicted_price', 0)
        change_pct = r.get('best_price_change_pct', 0)
        change = pred_price - r['latest_close']
        print(f"{symbol:8s} {r['latest_close']:14.2f} {pred_price:16.2f} {change:+10.2f} {change_pct:+8.2f}% "
              f"{r.get('best_reg_model', 'N/A'):>20s}")

# Chi tiết từng model
print(f"\n{'='*110}")
print("CHI TIẾT DỰ BÁO TỪNG MODEL")
print("=" * 110)

for symbol in STOCK_SYMBOLS:
    r = all_results[symbol]
    if r['status'] != 'SUCCESS':
        continue
    
    print(f"\n--- {symbol} (Giá hiện tại: {r['latest_close']:.2f}) ---")
    
    # Classification details
    if 'classification' in r:
        print(f"  Phân loại (xu hướng t+1):")
        for model_name, pred in r['classification'].items():
            marker = "★" if model_name == r.get('best_cls_model') else " "
            print(f"    {marker} {model_name:25s} → {pred['trend']:5s} | P(Tăng)={pred['prob_up']:.4f} | P(Giảm)={pred['prob_down']:.4f} | Confidence={pred['confidence']:.4f}")
    
    # Regression details
    if 'regression' in r:
        print(f"  Hồi quy (giá đóng cửa t+1):")
        for model_name, pred in r['regression'].items():
            marker = "★" if model_name == r.get('best_reg_model') else " "
            print(f"    {marker} {model_name:25s} → {pred['predicted_price']:.2f} ({pred['price_change_pct']:+.2f}%)")

## 6.7. Lưu kết quả dự báo ra file CSV

Lưu kết quả inference với đầy đủ thông tin: **timestamp**, **mã CP**, **giá hiện tại**, **xu hướng dự báo**, **xác suất**, **giá dự báo t+1**. File này có thể được append hàng ngày để theo dõi lịch sử dự báo.

In [ ]:
# === LƯU KẾT QUẢ DỰ BÁO RA CSV ===
output_rows = []

for symbol in STOCK_SYMBOLS:
    r = all_results[symbol]
    if r['status'] != 'SUCCESS':
        continue
    
    row = {
        'timestamp': r['timestamp'],
        'ticker': symbol,
        'latest_date': r['latest_date'],
        'latest_close': r['latest_close'],
        'latest_volume': r['latest_volume'],
        # Best Classification
        'best_cls_model': r.get('best_cls_model', ''),
        'predicted_trend': r.get('best_trend', ''),
        'prob_up': r.get('best_prob_up', 0),
        'confidence': r.get('best_confidence', 0),
        'ensemble_trend': r.get('ensemble_trend', ''),
        'ensemble_agreement': r.get('ensemble_agreement', ''),
        # Best Regression
        'best_reg_model': r.get('best_reg_model', ''),
        'predicted_price_t1': r.get('best_predicted_price', 0),
        'price_change_pct': r.get('best_price_change_pct', 0),
    }
    
    # Thêm chi tiết từng model classification
    if 'classification' in r:
        for model_name, pred in r['classification'].items():
            row[f'{model_name}_trend'] = pred['trend']
            row[f'{model_name}_prob_up'] = pred['prob_up']
    
    # Thêm chi tiết từng model regression
    if 'regression' in r:
        for model_name, pred in r['regression'].items():
            row[f'{model_name}_price'] = pred['predicted_price']
            row[f'{model_name}_change_pct'] = pred['price_change_pct']
    
    output_rows.append(row)

# Tạo DataFrame kết quả
inference_df = pd.DataFrame(output_rows)

# Lưu file CSV mới (theo ngày)
today_str = datetime.now().strftime('%Y%m%d')
output_path = os.path.join(OUTPUTS_DIR, f'inference_{today_str}.csv')
inference_df.to_csv(output_path, index=False)
print(f"✓ Kết quả dự báo → {output_path}")

# Append vào file lịch sử (tổng hợp tất cả các ngày chạy inference)
history_path = os.path.join(OUTPUTS_DIR, 'inference_history.csv')
if os.path.exists(history_path):
    # Đọc file cũ và append
    history_df = pd.read_csv(history_path)
    history_df = pd.concat([history_df, inference_df], ignore_index=True)
else:
    history_df = inference_df

history_df.to_csv(history_path, index=False)
print(f"✓ Lịch sử dự báo → {history_path} ({len(history_df)} dòng tổng)")

# Hiển thị file output
print(f"\n--- Nội dung file inference_{today_str}.csv ---")
print(inference_df.to_string(index=False))

## 6.8. Trực quan hóa kết quả dự báo

In [ ]:
# === BIỂU ĐỒ DỰ BÁO — Dashboard ===
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle(f'Dashboard Dự Báo Phiên T+1 — {datetime.now().strftime("%Y-%m-%d")}', 
             fontsize=18, fontweight='bold')

for idx, symbol in enumerate(STOCK_SYMBOLS):
    ax = axes.flatten()[idx]
    r = all_results[symbol]
    
    if r['status'] != 'SUCCESS':
        ax.text(0.5, 0.5, f'{symbol}\nERROR', transform=ax.transAxes, ha='center', fontsize=16)
        continue
    
    # Vẽ bar chart: xác suất Tăng/Giảm cho từng model
    model_names_short = ['LR', 'RF', 'XGB']
    prob_ups = [r['classification'][m]['prob_up'] for m in CLF_MODELS if m in r['classification']]
    prob_downs = [r['classification'][m]['prob_down'] for m in CLF_MODELS if m in r['classification']]
    
    x = np.arange(len(model_names_short))
    width = 0.35
    
    bars_up = ax.bar(x - width/2, prob_ups, width, label='P(Tăng)', color='green', alpha=0.7)
    bars_down = ax.bar(x + width/2, prob_downs, width, label='P(Giảm)', color='red', alpha=0.7)
    
    # Hiển thị giá trị
    for bar, val in zip(bars_up, prob_ups):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01, 
                f'{val:.2f}', ha='center', fontsize=8)
    for bar, val in zip(bars_down, prob_downs):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01, 
                f'{val:.2f}', ha='center', fontsize=8)
    
    trend_color = 'green' if r.get('best_trend') == 'TANG' else 'red'
    pred_price = r.get('best_predicted_price', 0)
    change_pct = r.get('best_price_change_pct', 0)
    
    ax.set_title(f"{symbol} | Giá: {r['latest_close']:.1f} → {pred_price:.1f} ({change_pct:+.1f}%)\n"
                 f"Dự báo: {r.get('best_trend', 'N/A')} | Ensemble: {r.get('ensemble_trend', 'N/A')}", 
                 fontsize=11, fontweight='bold', color=trend_color)
    ax.set_xticks(x)
    ax.set_xticklabels(model_names_short)
    ax.set_ylabel('Xác suất')
    ax.set_ylim(0, 1.1)
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2, axis='y')

# Ẩn subplot thừa
for idx in range(len(STOCK_SYMBOLS), len(axes.flatten())):
    axes.flatten()[idx].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, f'inference_dashboard_{today_str}.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Dashboard → outputs/inference_dashboard_{today_str}.png")

## Tổng kết Bước 6 — Suy Luận & Cập Nhật (Inference)

### Pipeline suy luận tự động hoàn chỉnh:
1. **Tải dữ liệu mới nhất** — vnstock (fallback yfinance) cho 5 mã + VN-Index + USD/VND
2. **Tính toán features** — 29 features (Technical Indicators, Lag, Volatility, Market)
3. **Chạy 5 models** — 3 Classification + 2 Regression cho mỗi mã
4. **Tổng hợp** — Best model + Ensemble voting
5. **Xuất kết quả** — CSV (timestamp, mã CP, xác suất, giá dự báo) + Dashboard PNG

### Output files:
- `outputs/inference_YYYYMMDD.csv` — Dự báo ngày hôm nay
- `outputs/inference_history.csv` — Lịch sử tất cả dự báo (append mỗi ngày)
- `outputs/inference_dashboard_YYYYMMDD.png` — Dashboard trực quan

### Hàm core:
```python
daily_inference(ticker, model, scaler) -> dict
```
- **Input**: mã CP, dict models, scaler
- **Output**: dict chứa xu hướng, xác suất, giá dự báo t+1, confidence, ensemble vote

### Cách sử dụng hàng ngày:
Chỉ cần **chạy lại toàn bộ notebook** sau phiên giao dịch kết thúc → tự động tải data mới, tính features, dự báo và lưu kết quả.